In [51]:
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path

ANNOTATIONS_DIR = Path("Annotations")
TIMESTAMPS_DIR  = Path("Timestamps")
EVAL_DIR        = Path("Evaluations_MR")
EVAL_DIR.mkdir(exist_ok=True)

VIDEO_IDS = [25, 26, 27, 28]

In [52]:
# ═══════════════════════════════════════════════════════════════
# SECTION 2 — Parsers (run once)
# ═══════════════════════════════════════════════════════════════

def parse_timestamp_str(ts_str):
    ts_str = ts_str.strip()
    parts  = ts_str.split(":")
    return int(parts[0]) * 60 + float(parts[1])


def parse_timestamp_range(ts_range):
    ts_range = re.sub(r"\s*-\s*", "-", ts_range.strip())
    parts    = re.split(r"-(?=\d{2}:)", ts_range)
    if len(parts) != 2:
        raise ValueError(f"Cannot parse: '{ts_range}'")
    return parse_timestamp_str(parts[0]), parse_timestamp_str(parts[1])


def load_annotations(video_id):
    for ext in ["csv", "txt"]:
        f = ANNOTATIONS_DIR / f"GroundTruthVideo{video_id}.{ext}"
        if f.exists():
            csv_file = f
            break
    else:
        print(f"  Warning: no annotation file for video_{video_id}")
        return []

    df = pd.read_csv(csv_file)
    annotations = []
    for _, row in df.iterrows():
        try:
            start, end = parse_timestamp_range(str(row["timestamp"]))
            annotations.append({
                "annotation":   row["annotation"],
                "start":        start,
                "end":          end,
                "subjectivity": int(row["subjectivity_rating"]),
            })
        except Exception as e:
            print(f"  Warning: skipping row — {e}")
    return annotations


def parse_ugly_json(video_id):
    json_file = TIMESTAMPS_DIR / f"video_{video_id}_with_predictions.json"
    if not json_file.exists():
        print(f"  Warning: no predictions file for video_{video_id}")
        return []

    with open(json_file, "r") as f:
        raw = json.load(f)

    n = len(raw["prompt_type"])
    results = []
    for i in range(n):
        idx = str(i)
        def parse_ts_list(ts_list):
            return [(t[0], t[1]) for t in ts_list]
        results.append({
            "prompt_type":     raw["prompt_type"][idx],
            "vlm_source":      raw["vlm_source"][idx],
            "vlm_frame_count": raw["vlm_frame_count"][idx],
            "queries":         raw["parsed_events"][idx],
            "chunks":          parse_ts_list(raw["predicted_timestamps_chunks"][idx]),
            "sped_up":         parse_ts_list(raw["predicted_timestamps_sped_up"][idx]),
        })
    return results


def make_combo_key(prompt_type, vlm_source, vlm_frame_count):
    return f"vlm={vlm_source}__frames={vlm_frame_count}__llm={prompt_type}"


print("✅ Parsers loaded")

✅ Parsers loaded


In [53]:
# ═══════════════════════════════════════════════════════════════
# SECTION 3 — Metrics (run once)
# ═══════════════════════════════════════════════════════════════

def merge_intervals(intervals):
    """Merge overlapping (start, end) intervals."""
    if not intervals:
        return []
    sorted_ivs = sorted(intervals, key=lambda x: x[0])
    merged = [list(sorted_ivs[0])]
    for start, end in sorted_ivs[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return [(s, e) for s, e in merged]


def total_duration(intervals):
    return sum(e - s for s, e in intervals)


def interval_intersection(ivs_a, ivs_b):
    """Total duration of intersection between two sets of intervals."""
    total = 0.0
    for a_start, a_end in ivs_a:
        for b_start, b_end in ivs_b:
            total += max(0.0, min(a_end, b_end) - max(a_start, b_start))
    return total


def coverage_metrics(annotations, predicted_timestamps, subjectivity_threshold=None):
    """
    Compute total coverage of GT annotations by predictions.

    subjectivity_threshold — if set, only include annotations with
                             subjectivity >= threshold as ground truth
    """
    if subjectivity_threshold is not None:
        ann_filtered = [a for a in annotations if a["subjectivity"] >= subjectivity_threshold]
    else:
        ann_filtered = annotations

    if not ann_filtered:
        return None

    gt_intervals   = merge_intervals([(a["start"], a["end"]) for a in ann_filtered])
    pred_intervals = merge_intervals(list(predicted_timestamps))

    gt_duration   = total_duration(gt_intervals)
    pred_duration = total_duration(pred_intervals)
    intersection  = interval_intersection(gt_intervals, pred_intervals)

    coverage  = intersection / gt_duration   if gt_duration   > 0 else 0.0
    precision = intersection / pred_duration if pred_duration > 0 else 0.0
    f1 = (2 * coverage * precision / (coverage + precision)
          if (coverage + precision) > 0 else 0.0)

    return {
        "gt_duration_sec":    round(gt_duration,  2),
        "pred_duration_sec":  round(pred_duration, 2),
        "intersection_sec":   round(intersection,  2),
        "coverage":           round(coverage,  4),
        "precision":          round(precision, 4),
        "f1":                 round(f1,        4),
        "n_gt_annotations":   len(ann_filtered),
        "subjectivity_filter": subjectivity_threshold,
    }


print("✅ Metrics loaded")

✅ Metrics loaded


In [54]:
# ═══════════════════════════════════════════════════════════════
# SECTION 4 — Save / skip helpers (run once)
# ═══════════════════════════════════════════════════════════════

def already_evaluated(video_id, combo_key, strategy):
    eval_file = EVAL_DIR / f"video_{video_id}_mr_eval.json"
    if not eval_file.exists():
        return False
    with open(eval_file, "r") as f:
        data = json.load(f)
    return any(
        e["combo_key"] == combo_key and e["mr_strategy"] == strategy
        for e in data.get("evaluations", [])
    )


def save_eval(video_id, combo_key, prompt_type, vlm_source,
              vlm_frame_count, strategy, metrics_all,
              metrics_important, metrics_critical):
    eval_file = EVAL_DIR / f"video_{video_id}_mr_eval.json"
    data = {"video_id": video_id, "evaluations": []}
    if eval_file.exists():
        with open(eval_file, "r") as f:
            data = json.load(f)

    data["evaluations"].append({
        "combo_key":       combo_key,
        "vlm_prompt_type": vlm_source,
        "vlm_frame_count": vlm_frame_count,
        "llm_prompt_type": prompt_type,
        "mr_strategy":     strategy,
        # three views of the same predictions vs different GT filters
        "metrics_all":       metrics_all,        # all annotations
        "metrics_important": metrics_important,  # subjectivity >= 1
        "metrics_critical":  metrics_critical,   # subjectivity >= 2 (objectively essential)
    })

    with open(eval_file, "w") as f:
        json.dump(data, f, indent=2)
    print(f"    💾 Saved → {eval_file}")


print("✅ Save helpers loaded")

✅ Save helpers loaded


In [55]:
# ═══════════════════════════════════════════════════════════════
# SECTION 5 — Pipeline (re-run after any new predictions)
# ═══════════════════════════════════════════════════════════════

for video_id in VIDEO_IDS:
    print(f"\n{'='*60}\n  video_{video_id}\n{'='*60}")

    annotations = load_annotations(video_id)
    if not annotations:
        print("  No annotations, skipping")
        continue

    mr_results = parse_ugly_json(video_id)
    if not mr_results:
        print("  No predictions, skipping")
        continue

    for result in mr_results:
        combo_key = make_combo_key(
            result["prompt_type"],
            result["vlm_source"],
            result["vlm_frame_count"]
        )
        print(f"\n  Combo: {combo_key}")

        for strategy in ["chunks", "sped_up"]:
            if already_evaluated(video_id, combo_key, strategy):
                print(f"    ⏭️  {strategy} — already done")
                continue

            predicted = result[strategy]

            m_all       = coverage_metrics(annotations, predicted)
            m_important = coverage_metrics(annotations, predicted, subjectivity_threshold=1)
            m_critical  = coverage_metrics(annotations, predicted, subjectivity_threshold=2)

            save_eval(video_id, combo_key, result["prompt_type"],
                      result["vlm_source"], result["vlm_frame_count"],
                      strategy, m_all, m_important, m_critical)

            print(f"    ✅ {strategy}")
            print(f"       all        → coverage={m_all['coverage']:.3f} | "
                  f"precision={m_all['precision']:.3f} | f1={m_all['f1']:.3f}")
            if m_important:
                print(f"       important  → coverage={m_important['coverage']:.3f} | "
                      f"precision={m_important['precision']:.3f} | f1={m_important['f1']:.3f}")
            if m_critical:
                print(f"       critical   → coverage={m_critical['coverage']:.3f} | "
                      f"precision={m_critical['precision']:.3f} | f1={m_critical['f1']:.3f}")

print("\n\n✅ All done!")


  video_25

  Combo: vlm=long__frames=30__llm=concise
    💾 Saved → Evaluations_MR\video_25_mr_eval.json
    ✅ chunks
       all        → coverage=0.164 | precision=0.312 | f1=0.215
       important  → coverage=0.370 | precision=0.312 | f1=0.339
       critical   → coverage=0.000 | precision=0.000 | f1=0.000
    💾 Saved → Evaluations_MR\video_25_mr_eval.json
    ✅ sped_up
       all        → coverage=0.164 | precision=0.312 | f1=0.215
       important  → coverage=0.370 | precision=0.312 | f1=0.339
       critical   → coverage=0.000 | precision=0.000 | f1=0.000

  Combo: vlm=long__frames=30__llm=detailed
    💾 Saved → Evaluations_MR\video_25_mr_eval.json
    ✅ chunks
       all        → coverage=0.262 | precision=0.404 | f1=0.318
       important  → coverage=0.370 | precision=0.252 | f1=0.300
       critical   → coverage=0.000 | precision=0.000 | f1=0.000
    💾 Saved → Evaluations_MR\video_25_mr_eval.json
    ✅ sped_up
       all        → coverage=0.262 | precision=0.404 | f1=0.318
   

In [56]:
# ═══════════════════════════════════════════════════════════════
# SECTION 6 — Summary table & rankings
# ═══════════════════════════════════════════════════════════════

def load_all_mr_evals():
    rows = []
    for video_id in VIDEO_IDS:
        eval_file = EVAL_DIR / f"video_{video_id}_mr_eval.json"
        if not eval_file.exists():
            continue
        with open(eval_file, "r") as f:
            data = json.load(f)
        for e in data.get("evaluations", []):
            for filter_name, metrics_key in [
                ("all",       "metrics_all"),
                ("important", "metrics_important"),
                ("critical",  "metrics_critical"),
            ]:
                m = e.get(metrics_key)
                if not m:
                    continue
                rows.append({
                    "video_id":        video_id,
                    "combo_key":       e["combo_key"],
                    "vlm_prompt_type": e["vlm_prompt_type"],
                    "vlm_frame_count": e["vlm_frame_count"],
                    "llm_prompt_type": e["llm_prompt_type"],
                    "mr_strategy":     e["mr_strategy"],
                    "filter":          filter_name,
                    "coverage":        m["coverage"],
                    "precision":       m["precision"],
                    "f1":              m["f1"],
                    "gt_duration":     m["gt_duration_sec"],
                    "pred_duration":   m["pred_duration_sec"],
                    "intersection":    m["intersection_sec"],
                })
    return pd.DataFrame(rows)


mr_df = load_all_mr_evals()

if mr_df.empty:
    print("No evaluations yet — run Section 5 first")
else:
    for filter_name in ["all", "important", "critical"]:
        subset = mr_df[mr_df["filter"] == filter_name]
        if subset.empty:
            continue
        ranked = (
            subset.groupby(["vlm_prompt_type", "vlm_frame_count",
                            "llm_prompt_type", "mr_strategy"])
            .agg(
                avg_coverage  =("coverage",  "mean"),
                avg_precision =("precision", "mean"),
                avg_f1        =("f1",        "mean"),
                n_videos      =("video_id",  "count"),
            )
            .reset_index()
            .sort_values("avg_f1", ascending=False)
        )
        ranked.index = range(1, len(ranked) + 1)
        print(f"\n=== Rankings — GT filter: {filter_name} ===\n")
        print(ranked.to_string())
        best = ranked.iloc[0]
        print(f"\n🏆 Best F1: vlm={best['vlm_prompt_type']} | "
              f"frames={best['vlm_frame_count']} | "
              f"llm={best['llm_prompt_type']} | "
              f"strategy={best['mr_strategy']}")


=== Rankings — GT filter: all ===

   vlm_prompt_type  vlm_frame_count llm_prompt_type mr_strategy  avg_coverage  avg_precision  avg_f1  n_videos
1             long               30        detailed      chunks        0.2623         0.4038  0.3180         1
2             long               30        detailed     sped_up        0.2623         0.4038  0.3180         1
3             long               30         concise      chunks        0.1639         0.3125  0.2150         1
4             long               30         concise     sped_up        0.1639         0.3125  0.2150         1
5           medium               30        detailed      chunks        0.1326         0.3429  0.1912         1
6           medium               30        detailed     sped_up        0.1326         0.3429  0.1912         1
7            short               30         concise      chunks        0.1740         0.1904  0.1818         1
8            short               30         concise     sped_up        0.174

In [57]:
# ═══════════════════════════════════════════════════════════════
# SECTION 7 — Inspector: drill into one combo
# Change these variables and re-run
# ═══════════════════════════════════════════════════════════════

INSPECT_VIDEO_ID = 25
INSPECT_VLM      = "long"
INSPECT_FRAMES   = 30
INSPECT_LLM      = "detailed"
INSPECT_STRATEGY = "chunks"

eval_file = EVAL_DIR / f"video_{INSPECT_VIDEO_ID}_mr_eval.json"
if not eval_file.exists():
    print("No eval file — run Section 5 first")
else:
    with open(eval_file, "r") as f:
        data = json.load(f)

    key    = make_combo_key(INSPECT_LLM, INSPECT_VLM, INSPECT_FRAMES)
    result = next(
        (e for e in data["evaluations"]
         if e["combo_key"] == key and e["mr_strategy"] == INSPECT_STRATEGY),
        None
    )

    if not result:
        print(f"Not found: {key} | {INSPECT_STRATEGY}\nAvailable:")
        for e in data["evaluations"]:
            print(f"  {e['combo_key']} | {e['mr_strategy']}")
    else:
        print(f"{'='*60}")
        print(f"  video_{INSPECT_VIDEO_ID} | {key} | {INSPECT_STRATEGY}")
        print(f"{'='*60}\n")

        for filter_name, metrics_key in [
            ("all annotations",       "metrics_all"),
            ("important (subj >= 1)", "metrics_important"),
            ("critical  (subj >= 2)", "metrics_critical"),
        ]:
            m = result.get(metrics_key)
            if not m:
                continue
            print(f"  [{filter_name}]  n={m['n_gt_annotations']} events | "
                  f"GT={m['gt_duration_sec']}s | pred={m['pred_duration_sec']}s | "
                  f"overlap={m['intersection_sec']}s")
            print(f"    coverage={m['coverage']:.3f}  "
                  f"precision={m['precision']:.3f}  "
                  f"f1={m['f1']:.3f}\n")

  video_25 | vlm=long__frames=30__llm=detailed | chunks

  [all annotations]  n=10 events | GT=61.0s | pred=39.63s | overlap=16.0s
    coverage=0.262  precision=0.404  f1=0.318

  [important (subj >= 1)]  n=4 events | GT=27.0s | pred=39.63s | overlap=10.0s
    coverage=0.370  precision=0.252  f1=0.300

  [critical  (subj >= 2)]  n=2 events | GT=6.0s | pred=39.63s | overlap=0.0s
    coverage=0.000  precision=0.000  f1=0.000

